Ce notebook permet de lancer un LDA uniquement sur les titres du sommaire et nn plus tout le contenu. 
Le but est d'obtenir des résultats mons bruités avec un sommaire qui porte déjà l'info nécessaire. 

In [3]:
!uv pip install -q nltk gensim pyLDAvis unidecode matplotlib seaborn pandas pyarrow

In [4]:
!uv pip install -q langchain-huggingface==0.0.3

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, SnowballStemmer
from nltk.stem.porter import *
from nltk.tokenize import word_tokenize
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
import html 
import json
import matplotlib.pyplot  as plt
import nltk
import numpy as np
import os
import pandas as pd
import unidecode
import re
import requests
import seaborn as sns
import string
import unidecode
import warnings
import torch
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to /home/onyxia/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/onyxia/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/onyxia/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [6]:
import tqdm as notebook_tqdm

In [7]:
class CachedLemmatizer:
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        self.cache = {}  # Manual cache as a dictionary

    def lemmatize(self, word, pos='n'):
        if word in self.cache:
            return self.cache[word]
        else:
            lemmatized_word = self.lemmatizer.lemmatize(word, pos)
            self.cache[word] = lemmatized_word  # Store in cache
            return lemmatized_word


cached_lemmatizer = CachedLemmatizer()

In [8]:
if (torch.cuda.is_available()):
    DEVICE="cuda"
else:
    DEVICE="cpu"

In [9]:
stop_words = set(stopwords.words('french'))

In [10]:
OUTPUT_DIR="intermediate_data"

# Utils preprocessing

In [17]:
def preprocess_text(text, lang="french"):
    # décoage HTML
    text = html.unescape(text)
    # nettoyage de tous les cractères spéciaux
    text = re.sub(r"&[a-z]+;", " ", text)
    text = re.sub(r"&#\d+;", " ", text)
    text = re.sub(r"[<>{}\[\]\|\^\~`\"'=]+", " ", text)
    text = re.sub(r"[–—•«»]+", " ", text)  # Tirets longs, puces, guillemets français
    text = re.sub(r"\.{2,}", " ", text) 

    # tokenisation
    words = word_tokenize(text)

    # lemming
    #stemmer = SnowballStemmer(lang)
                  
    wnl = cached_lemmatizer
   
    words_cleaned = []
    for w in words:
        #w_norm = unidecode.unidecode(w.lower())
        w_norm = w.lower()
        if (
            w_norm not in stop_words
            and w_norm not in string.punctuation
            and not re.search(r"[<>]|--+|__+|xx+|==+", w_norm)
            and len(w_norm) > 2
        ):
            words_cleaned.append(wnl.lemmatize(w_norm))
            #words_cleaned.append(stemmer.stem(w_norm))

    return words_cleaned


In [21]:
#exemple 
text = "Révision-- de l&rsquo;accord : &gt;&gt;' Tous les deux ans, les partenaires sociaux se réunissent. << Suivi de l’accord........ "
print(preprocess_text(text))

['révision', 'accord', 'tous', 'deux', 'an', 'partenaires', 'sociaux', 'réunissent', 'suivi', 'accord']


In [22]:
text = """
ARTICLE 3.6. Contingent annuel d’heures supplémentaires

Article 3.6.1. Fixation du contingent annuel d’heures supplémentaires

Les parties au présent accord conviennent de fixer le contingent annuel d’heures supplémentaires à 500 heures au jour de la signature du présent accord.

"""

print(preprocess_text(text))

['article', '3.6', 'contingent', 'annuel', 'heures', 'supplémentaires', 'article', '3.6.1', 'fixation', 'contingent', 'annuel', 'heures', 'supplémentaires', 'party', 'présent', 'accord', 'conviennent', 'fixer', 'contingent', 'annuel', 'heures', 'supplémentaires', '500', 'heures', 'jour', 'signature', 'présent', 'accord']


In [23]:
model_kwargs = {'device': DEVICE}  
#MODEL_NAME_EMBEDDER="BAAI/bge-small-en-v1.5"  
MODEL_NAME_EMBEDDER="BAAI/bge-m3" #gros modèle multilingue

embedder = HuggingFaceEmbeddings(
    model_name=MODEL_NAME_EMBEDDER, 
    model_kwargs=model_kwargs,
    show_progress=False
)


phrases_non_metier = [
    "Révision de l’accord",
    "Dénonciation de l’accord",
    "Interprétation de l’accord",
    "Suivi de l’accord",
    "Durée de l’accord",
    "Formalités de publicité et de dépôt",
    "Publicité et dépôt",
    "Date d'effet et durée",
    "Champ d'application",
    "Clause de revoyure", 
    "Information des représentants du personnel", 
    "Dispositions relatives à l’accord",
    "Champ d’application",
    "Commission de suivi", 
    "Pause déjeuner du personnel", 
    "Modification de l'accord",
    "Adhésion", 
    "Information du Comité Social et Economique", 
    "Annexe", 
    "Dispositions finales", 
    "Salariés concernés"
    
]

# Embeddings des phrases non-métier
ref_embeddings = embedder.embed_documents(phrases_non_metier)


def filtre_par_similarite_vectorise(titres, seuil=0.85):
    if not phrases:
        return []
    phrases =[html.unescape(phrase) for phrase in phrases]
    phrases = [re.sub(r"(?im)^article\s+\d+(\.\d+)*\s*[\.\-\–\:]*\s*", "", phrase) for phrase in phrases]
    phrase_embeddings = embedder.embed_documents(phrases)  
    sims = cosine_similarity(phrase_embeddings, ref_embeddings)

    # On garde les phrases dont la similarité max avec une phrase non-métier est < seuil
    keep_idx = np.max(sims, axis=1) < seuil
    return [phrase for phrase, keep in zip(phrases, keep_idx) if keep]

    
def filtre_sommaire_par_titre(summary, phrases_non_metier, seuil=0.85): #seuil arbitraire : en tester plsr
    """
    Ne garde que les chunks dont le titre est peu similaire aux phrases non métier.
    """
    if not summary:
        return []

    summary = [re.sub(r"(?im)^\s*(article\s+\d+(?:[\.\-]\d+)*|titre\s+[ivxlcdm]+)\s*[\.\-\–\:]*\s*","",
        titre.strip(),
        flags=re.IGNORECASE,
    )
    for titre in summary
    ]
    summary =[html.unescape(titre) for titre in summary]
    summary = [re.sub(r"\xa0", " ", titre) for titre in summary]

    # Embeddings des titres de section
    titre_embeddings = embedder.embed_documents(summary)
    ref_embeddings = embedder.embed_documents(phrases_non_metier)

    sims = cosine_similarity(titre_embeddings, ref_embeddings)
    
    # On garde les phrases dont la similarité max avec une phrase non-métier est < seuil
    keep_idx = np.max(sims, axis=1) < seuil
    return [phrase for phrase, keep in zip(phrases, keep_idx) if keep]


/home/onyxia/work/TopicModeling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Pour HS

In [24]:
sommaire_hs = pd.read_parquet("data/echantillon_1000_hs_accords_TOC.parquet")
df_hs = pd.read_parquet("data/echantillon_1000_hs_accords.parquet")
df_hs = df_hs.set_index("numdossier_new")
df_hs = df_hs.merge(sommaire_hs,how="inner",left_index=True,right_index=True)
df_hs = df_hs.rename(columns={"extracted_summary":"summary"})

In [25]:
def nettoyer_titres(titres):
    """
    Nettoyer les titres pour ne garder que l'information du titre en lui même et ne pas bruiter avec indications de sections (important quand on compare les embeddings) 
    Exmple :
    nettoyer_titres("ARTICLE 1 : OBJET DE L’ACCORD") = "OBJET DE L’ACCORD"
    """
    titres = [html.unescape(titre) for titre in titres]
    titres = [
        re.sub(
            r"""(?imx)                                     # modes : ignore case + verbose
            ^\s*                                           # début de ligne + espaces
            (                                              # groupe principal
                (?:
                    (article|titre|chapitre)               # mots-clés
                    \s+
                    (?:\d+(?:[\.\-]\d+)*|[ivxlcdm]+)        # numéros ou chiffres romains
                )
                |
                \d+(?:[\.\-\–]\d+)*\.?                     # ex : 2.1.2, 4–1
                |
                [a-zA-Z\d]{1,4}                             # ex : a, b, i, ii, 1, 2
                \)                                          # suivi d'une parenthèse fermante
            )
            \s*[\.\-\–\:]*\s*                              # séparateurs (., –, :, etc.)
            """,
            "",
            titre.strip()
        )
        for titre in titres
    ]


    # Nettoyage final (espaces, ponctuations parasites)
    titres = [
        titre.replace("\xa0", " ")
        .replace(" :", ":")
        .strip(" :\t\n")
        for titre in titres
    ]

    return titres

In [26]:
#exemple 
titres = ['PREAMBULE', 'ARTICLE 1 : OBJET DE L’ACCORD',
       'L’accord s’applique également aux intérimaires et salariés mis à disposition.',
       'ARTICLE 2 : CHAMP D’APPLICATION - BENEFICIAIRES',
       'ARTICLE 1 : LE TEMPS DE TRAVAIL EFFECTIF',
       'ARTICLE 2 : LE TEMPS DE PAUSE',
       'ARTICLE 3 : REPOS HEBDOMADAIRE ET QUOTIDIEN',
       'ARTICLE 4 : AMPLITUDE DE TRAVAIL',
       'L’employeur devra dès lors suivre la procédure suivante :',
       'ARTICLE 1 : PERIODE DE REFERENCE DU TEMPS DE TRAVAIL',
       'ARTICLE 2 : AMPLITUDE DES HORAIRES',
       'ARTICLE 3 : PROGRAMMATION ET MODALITES DE MODIFICATIONS DES HORAIRES',
       '- Délais de prévenance des modifications d’horaires de travail',
       'ARTICLE 4 : DECOMPTE ET PAIEMENT DES HEURES SUPPLEMENTAIRES',
       'ARTICLE 5 : MODALITES DE REMUNERATION',
       ': Lissage de la rémunération',
       ': Incidence des absences en cours de période de référence a) Rémunération des absences',
       'b) Décompte des heures',
       'Incidence des embauches ou départs en cours de période de référence',
       'Incidence des heures excédentaires et déficitaires en fin de période de référence',
       'ARTICLE 1 : ENTREE EN VIGUEUR ET DUREE DE L’ACCORD',
       'ARTICLE 2 : REVISION', 'ARTICLE 3 : DENONCIATION',
       'L’accord fera l’objet d’un affichage au siège de l’entreprise et dans le restaurant.']

titres = nettoyer_titres(titres)

In [27]:

def normalize(text):
    return unidecode.unidecode(text.lower().strip())

def filtre_titre(summary, phrases_non_metier, seuil=0.85): #seuil arbitraire : en tester plsr
    """
    Ne garde que les chunks dont le titre est peu similaire aux phrases non métier.
    """
    if not summary:
        return []

    summary = nettoyer_titres(summary)

    # Embeddings des titres de section
    titre_embeddings = embedder.embed_documents(summary)
    ref_embeddings = embedder.embed_documents(phrases_non_metier)

    sims = cosine_similarity(titre_embeddings, ref_embeddings)

    # On garde les chunks dont le titre est peu similaire aux phrases non métier
    keep_idx = np.max(sims, axis=1) < seuil
    titres_keep = [titre.strip() for titre, keep in zip(summary, keep_idx) if keep]
    processed_titres_keep = [preprocess_text(titre) for titre in titres_keep ]
    return(processed_titres_keep)




def get_valid_title_filtered(summary, skip_titles=["préambule", "annexe"], seuil_sim=0.85):
    skip_titles_norm = [normalize(t) for t in skip_titles]

    # supprimer le préambule et avant 
    preamble_idx = next((i for i, t in enumerate(summary) if "préambule" in normalize(t)), -1)
    if preamble_idx != -1:
        summary = summary[preamble_idx + 1:]

    # garder les titres valides uniquement
    valid_titles = [
        t for t in summary if all(skip_kw not in normalize(t) for skip_kw in skip_titles_norm)
    ]
    
    # filtrer par similarité des titres
    return filtre_titre(valid_titles, phrases_non_metier, seuil=seuil_sim)

In [28]:
df_hs["lda_documents"] = df_hs.apply(
    lambda row: get_valid_title_filtered(row["summary"]),
    axis=1
)


In [29]:
df_hs["lda_documents"].iloc[0]

[['champ', 'application', 'bénéficiaires'],
 ['rémunération', 'temp', 'travail'],
 ['prime', 'partage', 'valeur'],
 ['prime', 'ba', 'salaires'],
 ['récupération',
  'heures',
  'fériés',
  'fixation',
  'jour',
  'solidarité',
  '2023'],
 ['revalorisation', 'heures', 'supplémentaires'],
 ['prime', 'parrainage'],
 ['condition', 'travail'],
 ['utilisation', 'voitures', 'service']]

In [30]:
df_hs

,accorddocx,summary,lda_documents
numdossier_new,,,
T03323012553,Entre :\nL’Association d'Hospitalisation à Do...,"[Préambule : , Article 1 – Champ d’applicatio...","[[champ, application, bénéficiaires], [rémunér..."
T02719000708,\n ACCORD relatif à l’aménagement du temps de ...,[ ACCORD relatif à l’aménagement du temps de t...,"[[accord, relatif, aménagement, temp, travail,..."
T06719002035,\n\n\n\nENTRE LES SOUSSIGNES\n\n\nLa société B...,"[PREAMBULE , ARTICLE 1 : CHAMP D’APPLICATION, ...","[[champ, application], [object, accord], [depô..."
T01422005631,\n\nACCORD COLLECTIF D’ENTREPRISE RELATIF A L’...,[ACCORD COLLECTIF D’ENTREPRISE RELATIF A L’AUG...,"[[accord, collectif, entreprise, relatif, augm..."
T07118000137,\n\n\nLe présent accord est signé dans le resp...,"[Préambule , Article 1 – Champ d’application, ...","[[durée, travail], [aménagement, temp, travail..."
...,...,...,...
T03321008847,\n\n\n ACCORD COLLECTIF D’ENTREPRISE INORIX PR...,[ ACCORD COLLECTIF D’ENTREPRISE INORIX PROTECT...,"[[accord, collectif, entreprise, inorix, prote..."
T05123060220,ACCORD D’ENTREPRISE RELATIF A L’AUGMENTATION D...,[ACCORD D’ENTREPRISE RELATIF A L’AUGMENTATION ...,"[[accord, entreprise, relatif, augmentation, c..."
T08319001710,ACCORD D’ENTREPRISE RELATIF ...,[ACCORD D’ENTREPRISE RELATIF ...,"[[accord, entreprise, relatif, condition, trav..."


In [31]:
df_hs = df_hs.reset_index()
df_hs[["numdossier_new", "lda_documents"]].to_parquet(f"{OUTPUT_DIR}/processed_titles_hs.parquet", index=False)
df_hs[["numdossier_new", "lda_documents"]].to_csv(f"{OUTPUT_DIR}/processed_titles_hs.csv", index=False)
